In [1]:
from dotenv import load_dotenv
from IPython.display import display, Markdown
load_dotenv()

True

In [2]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
import boto3
from langchain_aws import ChatBedrock
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import create_react_agent

# Configurar el modelo con un ID válido
llm = ChatBedrock(
    model_id="amazon.nova-lite-v1:0",
    region_name="us-east-1",
    model_kwargs={
        "temperature": 0.5,
        "max_tokens": 2048,
        "top_p": 0.9,
    }
)

## Herramientas simples (MCP no funciona en Windows/Jupyter)

In [4]:
from langchain.tools import tool
from datetime import datetime
import pytz

@tool
def get_current_time(timezone: str = "UTC") -> str:
    """Get the current time in the specified timezone"""
    try:
        tz = pytz.timezone(timezone)
        current_time = datetime.now(tz)
        return current_time.strftime("%Y-%m-%d %H:%M:%S %Z")
    except:
        return datetime.now().strftime("%Y-%m-%d %H:%M:%S UTC")

@tool
def calculate(expression: str) -> str:
    """Calculate mathematical expressions"""
    try:
        result = eval(expression)
        return str(result)
    except:
        return "Invalid expression"

tools = [get_current_time, calculate]

In [5]:
# Crear agente con herramientas simples
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant with access to time and calculation tools."
)

In [6]:
from langchain_core.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="What time is it?")]}
)

print(response['messages'][-1].content)

<thinking>The User has asked for the current time. I need to determine the timezone to provide the accurate time. I will first ask the User for their preferred timezone.</thinking>

Hi! To give you the most accurate time, could you please specify the timezone you are interested in? The default timezone is UTC.


In [7]:
response = agent.invoke(
    {"messages": [HumanMessage(content="Calculate 25 * 4 + 10")]}
)

print(response['messages'][-1].content)

The result of the calculation 25 * 4 + 10 is 110.
